In [1]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx # we will use it for visualization
import seaborn as sns
from scipy.optimize import linear_sum_assignment
from scipy.optimize import milp
from scipy import optimize
from scipy.sparse.csgraph import shortest_path

## Common parameters

Same problem parameters as Requirements 2 and 3 (defined once, before the environment).

In [2]:
# Problem parameters (same as Requirements 2 and 3)
n_campaigns = 3
n_users = 2000              # Number of rounds T
B = 300                     # Total budget
rho = B / n_users           # Budget per round

# Valuation of each campaign
my_valuations = np.array([0.6, 0.7, 0.8])

# Discrete set of possible bids
available_bids = np.linspace(0, 1, 11)

# Conflict graph (same as Requirement 2): campaigns 2 and 3 are mutually exclusive
conflict_edges = [(1, 2)]

t = np.arange(n_users)

print("Campaign valuations:", my_valuations)
print("Possible bids:", available_bids)
print("Budget per round:", rho)

Campaign valuations: [0.6 0.7 0.8]
Possible bids: [0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8 0.9 1. ]
Budget per round: 0.15


## Slightly Non-stationary Environment (Requirement 4)
**Same generator, with $k_i(t)$ changing slowly.** Keep the uniform bidders and the same construction of the highest competing bid,
$$
m_i(t) = \max_{j=1,\dots,k_i(t)} u_{ij}(t), \qquad u_{ij}(t) \sim U(0,1), \qquad m_i(t) \sim \text{Beta}(k_i(t), 1),
$$
so the win probability of bid $b$ in campaign $i$ is again $b^{\,k_i(t)}$. What changes with respect to Requirement 3 is the *schedule* of $k_i(t)$: instead of cycling every few rounds, it is piecewise constant on a small number of long intervals. The horizon $[0, T)$ is partitioned by breakpoints $0 = \tau_0 < \tau_1 < \dots < \tau_P = T$ into $P$ phases, and inside phase $p$ the number of competitors is fixed:
$$
k_i(t) = k_i^{(p)} \quad \text{for } t \in [\tau_{p-1}, \tau_p), \qquad p = 1, \dots, P.
$$
With $T = 2000$ we use $P = 4$ phases with breakpoints $\tau = (0, 500, 1100, 1500, 2000)$ and
$$
k_1 = (1, 3, 2, 1), \qquad k_2 = (3, 1, 5, 2), \qquad k_3 = (5, 6, 1, 3).
$$
So:
1. in phase 1 (rounds $[0, 500)$) campaign 1 has 1 competitor and is the one worth bidding on, while campaigns 2 and 3 are hard ($k = 3$ and $k = 5$);
2. in phase 2 (rounds $[500, 1100)$) the roles switch: campaign 2 becomes the easy one ($k = 1$), campaign 1 gets harder ($k = 3$) and campaign 3 is essentially hopeless ($k = 6$);
3. in phase 3 (rounds $[1100, 1500)$) campaign 3 becomes the easy one ($k = 1$) while campaign 2 is the hardest ($k = 5$);
4. in phase 4 (rounds $[1500, 2000)$) campaign 1 is again the easy one ($k = 1$) and the others are moderately hard ($k = 2$ and $k = 3$).

The values are chosen with the same logic as in Requirement 3 (a campaign with $k = 1$ faces a mean competing bid of $0.5 < v_i$ and is *worth bidding*; a campaign with $k \ge 3$ faces a mean competing bid above $v_i$ and is *hard*), so in every phase a different campaign is the profitable one and both the optimal bid within a campaign and the optimal split of the budget across campaigns change at each breakpoint. The breakpoints are shared by all campaigns, so the change points are well-defined events. The phases have unequal lengths (500, 600, 400, 500 rounds) on purpose: a sliding window tuned to one phase length is not automatically right for the others.

Within a phase the environment is a proper stochastic bandit: $m_i(t)$ is i.i.d. $\text{Beta}(k_i^{(p)}, 1)$ for $t$ in the phase, with fixed win probabilities $b^{\,k_i^{(p)}}$. This is what makes the environment only *slightly* non-stationary: with phases of 400-600 rounds, a learner that forgets the past (sliding window) or resets when it detects a change (change detector) has enough time to re-estimate the means and behave near-optimally inside each phase, which is impossible in Requirement 3, where the phases last 10-30 rounds. As in Requirement 3, the sequence $k_i(t)$ is generated once with a fixed seed and then treated as given. In code it is the same `n_competitors_t` array, filled from the breakpoint list instead of `(t // L) % len(kappa)`, followed by the same `m_t` construction.

**Benchmark.** Because the phases are long, the natural benchmark is no longer the best *fixed* strategy in hindsight but the best *policy* in hindsight: a clairvoyant that knows the distribution of each phase. It solves the Requirement 2 LP once per phase, with that phase's win probabilities $b^{\,k_i^{(p)}}$ and its own share of the budget (the same per-round rate $\rho = B/T$ throughout), and the benchmark is the sum of the phase optima. This is stronger than the fixed benchmark of Requirement 3 and is the one that makes sense here, since tracking the phases is actually achievable.

**Algorithms compared.** (i) Combinatorial-UCB with a sliding window of length $W$, which estimates each bid's statistics only from the last $W$ rounds; (ii) Combinatorial-UCB with a change detector (CUSUM on the per-arm rewards), which resets the statistics of an arm when the detector fires; (iii) the primal-dual method of Requirement 3, run as-is. The expected picture is the mirror image of Requirement 3: the windowed and change-detecting UCBs exploit the within-phase stationarity and beat primal-dual on this slow-changing sequence, while primal-dual, whose guarantee is adversarial, is the only one that was fine on the fast-changing one.

**Feedback.** The two UCB variants are run with their native bandit feedback (the learner only observes whether it won the auctions it entered), while primal-dual keeps the full feedback of Requirement 3 (it observes $m_i(t)$). We keep each algorithm with its native feedback and state this explicitly in the comparison.

In [3]:
# Reproducibility: the sequence is generated ONCE and then treated as fixed
# (from the learner's point of view it is a given sequence; within each phase
# it is an i.i.d. stochastic environment)
np.random.seed(42)

# Slightly non-stationary schedule of the number of competitors:
# the horizon is split into a few long phases by shared breakpoints, and
# inside phase p campaign i has a fixed number of competitors k_i^(p)
breakpoints = [0, 500, 1100, 1500, n_users]   # phase p = [tau_{p-1}, tau_p)
k_phases = [
    [1, 3, 2, 1],   # campaign 1: worth it -> hard -> medium -> worth it
    [3, 1, 5, 2],   # campaign 2: hard -> worth it -> hard -> medium
    [5, 6, 1, 3],   # campaign 3: hard -> hard -> worth it -> medium
]
n_phases = len(breakpoints) - 1
assert all(len(k) == n_phases for k in k_phases)

# k_i(t) = k_i^(p) for t in [tau_{p-1}, tau_p)
n_competitors_t = np.zeros((n_campaigns, n_users), dtype=int)
for p in range(n_phases):
    start, end = breakpoints[p], breakpoints[p + 1]
    for campaign in range(n_campaigns):
        n_competitors_t[campaign, start:end] = k_phases[campaign][p]

# m_i(t) = max of k_i(t) uniform bids  ->  Beta(k_i(t), 1)
k_max = max(max(k) for k in k_phases)
all_bids = np.random.uniform(0, 1, size=(n_campaigns, k_max, n_users))
m_t_slow = np.zeros((n_campaigns, n_users))
for campaign in range(n_campaigns):
    for round_t in range(n_users):
        k = n_competitors_t[campaign, round_t]
        m_t_slow[campaign, round_t] = all_bids[campaign, :k, round_t].max()

phase_lengths = np.diff(breakpoints)
print("Breakpoints:", breakpoints, "-> phase lengths:", phase_lengths.tolist())
print("Competitors per phase k_i^(p):", k_phases)
print("Shape of m_t_slow:", m_t_slow.shape)

# Expected utility of the best bid in each phase (justifies the choice of k_phases):
# in every phase a different campaign is the "worth it" one
print(f"\n{'phase':>5} {'rounds':>12} {'camp':>4} {'v_i':>5} {'k':>3} "
      f"{'E[m]':>6} {'best b':>7} {'P(win)':>7} {'utility':>8}")
for p in range(n_phases):
    rounds = f"[{breakpoints[p]}, {breakpoints[p + 1]})"
    for campaign in range(n_campaigns):
        v = my_valuations[campaign]
        k = k_phases[campaign][p]
        utility = (v - available_bids) * available_bids ** k
        j = utility.argmax()
        print(f"{p + 1:>5} {rounds:>12} {campaign + 1:>4} {v:>5.1f} {k:>3d} "
              f"{k / (k + 1):>6.2f} {available_bids[j]:>7.1f} "
              f"{available_bids[j] ** k:>7.3f} {utility[j]:>8.4f}")

Breakpoints: [0, 500, 1100, 1500, 2000] -> phase lengths: [500, 600, 400, 500]
Competitors per phase k_i^(p): [[1, 3, 2, 1], [3, 1, 5, 2], [5, 6, 1, 3]]
Shape of m_t_slow: (3, 2000)

phase       rounds camp   v_i   k   E[m]  best b  P(win)  utility
    1     [0, 500)    1   0.6   1   0.50     0.3   0.300   0.0900
    1     [0, 500)    2   0.7   3   0.75     0.5   0.125   0.0250
    1     [0, 500)    3   0.8   5   0.83     0.7   0.168   0.0168
    2  [500, 1100)    1   0.6   3   0.75     0.4   0.064   0.0128
    2  [500, 1100)    2   0.7   1   0.50     0.3   0.300   0.1200
    2  [500, 1100)    3   0.8   6   0.86     0.7   0.118   0.0118
    3 [1100, 1500)    1   0.6   2   0.67     0.4   0.160   0.0320
    3 [1100, 1500)    2   0.7   5   0.83     0.6   0.078   0.0078
    3 [1100, 1500)    3   0.8   1   0.50     0.4   0.400   0.1600
    4 [1500, 2000)    1   0.6   1   0.50     0.3   0.300   0.0900
    4 [1500, 2000)    2   0.7   2   0.67     0.5   0.250   0.0500
    4 [1500, 2000)    3  